##Polars

In [17]:
import polars as pl
import pandas as pd
import bottleneck as bn
import numpy as np

In [18]:
df_polars = pl.read_csv('train.csv')
print(df_polars.head())

shape: (5, 12)
┌─────────────┬──────────┬────────┬───────────────────┬───┬───────────┬─────────┬───────┬──────────┐
│ PassengerId ┆ Survived ┆ Pclass ┆ Name              ┆ … ┆ Ticket    ┆ Fare    ┆ Cabin ┆ Embarked │
│ ---         ┆ ---      ┆ ---    ┆ ---               ┆   ┆ ---       ┆ ---     ┆ ---   ┆ ---      │
│ i64         ┆ i64      ┆ i64    ┆ str               ┆   ┆ str       ┆ f64     ┆ str   ┆ str      │
╞═════════════╪══════════╪════════╪═══════════════════╪═══╪═══════════╪═════════╪═══════╪══════════╡
│ 1           ┆ 0        ┆ 3      ┆ Braund, Mr. Owen  ┆ … ┆ A/5 21171 ┆ 7.25    ┆ null  ┆ S        │
│             ┆          ┆        ┆ Harris            ┆   ┆           ┆         ┆       ┆          │
│ 2           ┆ 1        ┆ 1      ┆ Cumings, Mrs.     ┆ … ┆ PC 17599  ┆ 71.2833 ┆ C85   ┆ C        │
│             ┆          ┆        ┆ John Bradley (Fl… ┆   ┆           ┆         ┆       ┆          │
│ 3           ┆ 1        ┆ 3      ┆ Heikkinen, Miss.  ┆ … ┆ STON/O2.  ┆ 7.92

In [24]:
print("Форма датасета", df_polars.shape)
print("\nТипы данных")
print(df_polars.dtypes)
print("\nОписательная статистика:")
print(df_polars.describe())
print("\nКоличество пропусков по столбцам:")
print(df_polars.null_count())

Форма датасета (891, 12)

Типы данных
[Int64, Int64, Int64, String, String, Float64, Int64, Int64, String, Float64, String, String]

Описательная статистика:
shape: (9, 13)
┌────────────┬─────────────┬──────────┬──────────┬───┬───────────┬───────────┬───────┬──────────┐
│ statistic  ┆ PassengerId ┆ Survived ┆ Pclass   ┆ … ┆ Ticket    ┆ Fare      ┆ Cabin ┆ Embarked │
│ ---        ┆ ---         ┆ ---      ┆ ---      ┆   ┆ ---       ┆ ---       ┆ ---   ┆ ---      │
│ str        ┆ f64         ┆ f64      ┆ f64      ┆   ┆ str       ┆ f64       ┆ str   ┆ str      │
╞════════════╪═════════════╪══════════╪══════════╪═══╪═══════════╪═══════════╪═══════╪══════════╡
│ count      ┆ 891.0       ┆ 891.0    ┆ 891.0    ┆ … ┆ 891       ┆ 891.0     ┆ 204   ┆ 889      │
│ null_count ┆ 0.0         ┆ 0.0      ┆ 0.0      ┆ … ┆ 0         ┆ 0.0       ┆ 687   ┆ 2        │
│ mean       ┆ 446.0       ┆ 0.383838 ┆ 2.308642 ┆ … ┆ null      ┆ 32.204208 ┆ null  ┆ null     │
│ std        ┆ 257.353842  ┆ 0.486592 ┆ 0.8

In [25]:
pclass_count = df_polars.group_by('Pclass').agg(pl.len().alias('count'))
print(pclass_count)

shape: (3, 2)
┌────────┬───────┐
│ Pclass ┆ count │
│ ---    ┆ ---   │
│ i64    ┆ u32   │
╞════════╪═══════╡
│ 1      ┆ 216   │
│ 2      ┆ 184   │
│ 3      ┆ 491   │
└────────┴───────┘


In [26]:
survived_by_gender = df_polars.filter(pl.col('Survived') == 1).group_by('Sex').agg(pl.len().alias('survived_count'))
print(survived_by_gender)

shape: (2, 2)
┌────────┬────────────────┐
│ Sex    ┆ survived_count │
│ ---    ┆ ---            │
│ str    ┆ u32            │
╞════════╪════════════════╡
│ female ┆ 233            │
│ male   ┆ 109            │
└────────┴────────────────┘


In [27]:
passengers_over_44 = df_polars.filter(pl.col('Age') > 44)
print(f"Количество пассажиров старше 44 лет: {len(passengers_over_44)}")
print(passengers_over_44.select(['PassengerId', 'Name', 'Age', 'Pclass', 'Sex']))

Количество пассажиров старше 44 лет: 115
shape: (115, 5)
┌─────────────┬─────────────────────────────────┬──────┬────────┬────────┐
│ PassengerId ┆ Name                            ┆ Age  ┆ Pclass ┆ Sex    │
│ ---         ┆ ---                             ┆ ---  ┆ ---    ┆ ---    │
│ i64         ┆ str                             ┆ f64  ┆ i64    ┆ str    │
╞═════════════╪═════════════════════════════════╪══════╪════════╪════════╡
│ 7           ┆ McCarthy, Mr. Timothy J         ┆ 54.0 ┆ 1      ┆ male   │
│ 12          ┆ Bonnell, Miss. Elizabeth        ┆ 58.0 ┆ 1      ┆ female │
│ 16          ┆ Hewlett, Mrs. (Mary D Kingcome… ┆ 55.0 ┆ 2      ┆ female │
│ 34          ┆ Wheadon, Mr. Edward H           ┆ 66.0 ┆ 2      ┆ male   │
│ 53          ┆ Harper, Mrs. Henry Sleeper (My… ┆ 49.0 ┆ 1      ┆ female │
│ …           ┆ …                               ┆ …    ┆ …      ┆ …      │
│ 858         ┆ Daly, Mr. Peter Denis           ┆ 51.0 ┆ 1      ┆ male   │
│ 863         ┆ Swift, Mrs. Frederick Joel 

##Ускорение работы с pandas

In [29]:
df_pandas = pd.read_csv('train.csv')
print(df_pandas.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [30]:
mean_age = bn.nanmean(df_pandas['Age'].values)
std_age = bn.nanstd(df_pandas['Age'].values)
print(f"Средний возраст: {mean_age:.2f}")
print(f"Стандартное отклонение возраста: {std_age:.2f}")

Средний возраст: 29.70
Стандартное отклонение возраста: 14.52


In [32]:
conditions = [
    df_pandas['Pclass'].isin([1, 2]),
    df_pandas['Pclass'] == 3
]


choices = [
    df_pandas['Fare'] * 1.3,
    df_pandas['Fare'] * 1.1
]


df_pandas['Fare_new'] = np.select(conditions, choices, default=df_pandas['Fare'])

##Оптимизация типов pandas

In [40]:
df = pd.read_csv('Housing.csv')

memory_before = df.memory_usage(deep=True).sum()
print(f"Потребление памяти до оптимизации: {memory_before} байт")

print("\nТекущие типы данных:")
print(df.dtypes)
print("\nИнформация о данных:")
df.info()

print("Тип данных каждого столбца")

print(f"\nprice:")
print(f"Текущий тип: {df['price'].dtype}")
print(f"Min: {df['price'].min()}, Max: {df['price'].max()}")
print(f"Стоит использовать int32")

print(f"\narea:")
print(f"Текущий тип: {df['area'].dtype}")
print(f"Min: {df['area'].min()}, Max: {df['area'].max()}")
print(f"Стоит использовать int32")

print(f"\nbedrooms:")
print(f"Текущий тип: {df['bedrooms'].dtype}")
print(f"Min: {df['bedrooms'].min()}, Max: {df['bedrooms'].max()}")
print(f"Уникальных значений: {df['bedrooms'].nunique()}")
print(f"Стоит использовать int8")

print(f"\nbathrooms:")
print(f"Текущий тип: {df['bathrooms'].dtype}")
print(f"Min: {df['bathrooms'].min()}, Max: {df['bathrooms'].max()}")
print(f"Уникальных значений: {df['bathrooms'].nunique()}")
print(f"Стоит использовать int8")

print(f"\nstories:")
print(f"Текущий тип: {df['stories'].dtype}")
print(f"Min: {df['stories'].min()}, Max: {df['stories'].max()}")
print(f"Уникальных значений: {df['stories'].nunique()}")
print(f"Стоит использовать int8")

binary_columns = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for col in binary_columns:
    print(f"\n{col}:")
    print(f"Текущий тип: {df[col].dtype}")
    print(f"Уникальные значения: {df[col].unique()}")
    print(f"Стоит использовать category")

print(f"\nparking:")
print(f"Текущий тип: {df['parking'].dtype}")
print(f"Min: {df['parking'].min()}, Max: {df['parking'].max()}")
print(f"Уникальных значений: {df['parking'].nunique()}")
print(f"Стоит использовать int8")

print(f"\nfurnishingstatus:")
print(f"Текущий тип: {df['furnishingstatus'].dtype}")
print(f"Уникальные значения: {df['furnishingstatus'].unique()}")
print(f"Стоит использовать category")

df_optimized = df.copy()

df_optimized['price'] = df_optimized['price'].astype('int32')
df_optimized['area'] = df_optimized['area'].astype('int32')
df_optimized['bedrooms'] = df_optimized['bedrooms'].astype('int8')
df_optimized['bathrooms'] = df_optimized['bathrooms'].astype('int8')
df_optimized['stories'] = df_optimized['stories'].astype('int8')
df_optimized['parking'] = df_optimized['parking'].astype('int8')

binary_columns = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for col in binary_columns:
    df_optimized[col] = df_optimized[col].astype('category')

df_optimized['furnishingstatus'] = df_optimized['furnishingstatus'].astype('category')

memory_after = df_optimized.memory_usage(deep=True).sum()

print(f"\nПотребление памяти до оптимизации: {memory_before} байт")
print(f"Потребление памяти после оптимизации: {memory_after} байт")


Потребление памяти до оптимизации: 227244 байт

Текущие типы данных:
price                int64
area                 int64
bedrooms             int64
bathrooms            int64
stories              int64
mainroad            object
guestroom           object
basement            object
hotwaterheating     object
airconditioning     object
parking              int64
prefarea            object
furnishingstatus    object
dtype: object

Информация о данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          54